## Collaborative filtering and recommendations

### At the end of this assignment, I should be able to
* Understand how item-item and user-user collaborative filtering perform recommendations
* Explain a experiment where we tested item-item versus user-user

In [9]:
import copy

# our standard imports
import numpy as np
import pandas as pd

# of course we need to be able to split into training and test
from sklearn.model_selection import train_test_split

def predict_user_user(data_raw,x_raw,N=10,frac=0.02):
    # data_raw is our uncentered data matrix. We want to make sure we drop the name of the user we
    # are predicting:
    db = data_raw.drop(x_raw.name)
    # We of course want to center and fill in missing values
    db = (db.T-db.T.mean()).fillna(0).T
    # Now this is a little tricky to think about, but we want to create a train test split of the movies
    # that user x_raw.name has rated. We need some of them but want some of them removed for testing.
    # This is where the frac parameter is used. I want you to think about how to select movies for training
    # Your solution here
    ix_raw, ix_raw_test = train_test_split(x_raw.dropna().index,test_size=frac,random_state=42) # Got to ignore some movies

    # Here is where we use what you figured out above
    x_raw_test = x_raw.loc[ix_raw_test]
    x_raw = x_raw.copy()
    x_raw.loc[ix_raw_test] = np.nan # ignore the movies in test
    x = (x_raw - x_raw.mean()).fillna(0)

    preds = []
    for movie in ix_raw_test:
        # Your solution here
        sims = db.loc[np.isnan(data_raw.drop(x_raw.name)[movie])==False].apply(lambda y: (y.values*x.values).sum()/(np.sqrt((y**2).sum())*np.sqrt((x**2).sum())),axis=1)
        sims = sims.dropna()
        try:
            sorted_sims = sims.sort_values()[::-1]
        except:
            preds.append(0) # means there is no one that also rated this movie amongst all other users
            continue
        top_sims = sorted_sims.iloc[:N]
        ids = top_sims.index
        # Your solution here
        preds.append(db.loc[ids, movie].mean())
    #print(preds)
    pred = pd.Series(preds,index=x_raw_test.index)
    #print(pred)
    actual = x_raw_test-x_raw.mean()
    #print(actual)
    mae = (actual-pred).abs().mean()
    return mae

def predict_item_item(data_raw,x_raw,N=10,frac=0.02,debug={}):
    ix_raw, ix_raw_test = train_test_split(x_raw.dropna().index,test_size=frac,random_state=42) # Got to ignore some movies
    x_raw_test = x_raw.loc[ix_raw_test]

    db = data_raw.drop(x_raw.name)
    db = (db.T-db.T.mean()).fillna(0).T
    db = db.T.fillna(0)
    preds = []
    for movie in ix_raw_test:
        x = db.loc[movie]
        sims = db.drop(movie).loc[ix_raw].apply(lambda y: (y.values*x.values).sum()/(np.sqrt((y**2).sum())*np.sqrt((x**2).sum())),axis=1)
        sims = sims.dropna()
        sorted_sims = sims.sort_values()[::-1]
        top_sims = sorted_sims.iloc[:N]
        ids = top_sims.index
        print(ids)
        preds.append(x_raw.loc[ids].mean())
    pred = pd.Series(preds,index=x_raw_test.index)
    actual = x_raw_test
    mae = (actual-pred).abs().mean()
    return mae


## Real dataset: Movielens

https://grouplens.org/datasets/movielens/

> MovieLens is a collaborative filtering system for movies. A
user of MovieLens rates movies using 1 to 5 stars, where 1 is "Awful" and 5 is "Must
See". MovieLens then uses the ratings of the community to recommend other movies
that user might be interested in, predict what that user might rate a movie,
or perform other tasks. - "Collaborative Filtering Recommender Systems"

In [1]:
import pandas as pd
import numpy as np

ratings = pd.read_csv(f'https://raw.githubusercontent.com/Anderson-Lab/csc-466-student/refs/heads/main/data/movielens-small/ratings.csv') # you might need to change this path
ratings = ratings.dropna()
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [2]:
movies = pd.read_csv(f'https://raw.githubusercontent.com/Anderson-Lab/csc-466-student/refs/heads/main/data/movielens-small/movies.csv')
movies = movies.dropna()
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### Joining the data together
We need to join those two source dataframes into a single one called data. I do this by setting the index to movieId and then specifying an ``inner`` join which means that the movie has to exist on both sides of the join. Then I reset the index so that I can later set the multi-index of userId and movieId. The results of this are displayed below. Pandas is awesome, but it takes some getting used to how everything works.

In [3]:
data = movies.set_index('movieId').join(ratings.set_index('movieId'),how='inner').reset_index()
data = data.drop('timestamp',axis=1) # We won't need timestamp here
data.head()

,movieId,title,genres,userId,rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5


In [4]:
ratings = data.set_index(['userId','movieId'])['rating']
ratings # as Series

userId  movieId
1       1          4.0
5       1          4.0
7       1          4.5
15      1          2.5
17      1          4.5
                  ... 
184     193581     4.0
        193583     3.5
        193585     3.5
        193587     3.5
331     193609     4.0
Name: rating, Length: 100836, dtype: float64

#### Exercise 1
I provide a structure for predicting recommentations using user-user collaborative filtering.  For this exercise, please complete the missing components.

``data_raw`` - your entire dataframe

``x_raw`` - the data from a single user

``N`` - neighborhood size

``frac`` - fraction for your test dataset

In [7]:
mae = predict_user_user(ratings.unstack(),ratings.unstack().loc[1])
mae

np.float64(0.8241596814667028)

#### Exercise 2
I provide a structure for predicting recommentations using item-item collaborative filtering. For this exercise, please complete the missing components.

In [10]:
mae = predict_item_item(ratings.unstack(),ratings.unstack().loc[1])
mae

/tmp/ipykernel_203/2974115222.py:60: RuntimeWarning: invalid value encountered in scalar divide
  sims = db.drop(movie).loc[ix_raw].apply(lambda y: (y.values*x.values).sum()/(np.sqrt((y**2).sum())*np.sqrt((x**2).sum())),axis=1)/2
/tmp/ipykernel_203/2974115222.py:60: RuntimeWarning: invalid value encountered in scalar divide
  sims = db.drop(movie).loc[ix_raw].apply(lambda y: (y.values*x.values).sum()/(np.sqrt((y**2).sum())*np.sqrt((x**2).sum())),axis=1)/2
/tmp/ipykernel_203/2974115222.py:60: RuntimeWarning: invalid value encountered in scalar divide
  sims = db.drop(movie).loc[ix_raw].apply(lambda y: (y.values*x.values).sum()/(np.sqrt((y**2).sum())*np.sqrt((x**2).sum())),axis=1)/2


Index([2028, 1210, 1198, 1222, 2571, 1196, 2959, 2542, 2329, 47], dtype='int64', name='movieId')
Index([3479, 1298, 1197, 804, 2641, 2616, 2094, 2291, 2406, 2872], dtype='int64', name='movieId')
Index([1445, 2096, 2193, 1676, 3243, 1805, 2143, 3450, 4006, 2033], dtype='int64', name='movieId')
Index([1552, 1573, 2253, 2450, 2054, 673, 648, 1644, 1804, 2273], dtype='int64', name='movieId')
Index([1196, 1210, 1198, 1291, 1270, 1197, 2571, 1208, 608, 2028], dtype='int64', name='movieId')


/tmp/ipykernel_203/2974115222.py:60: RuntimeWarning: invalid value encountered in scalar divide
  sims = db.drop(movie).loc[ix_raw].apply(lambda y: (y.values*x.values).sum()/(np.sqrt((y**2).sum())*np.sqrt((x**2).sum())),axis=1)/2
/tmp/ipykernel_203/2974115222.py:60: RuntimeWarning: invalid value encountered in scalar divide
  sims = db.drop(movie).loc[ix_raw].apply(lambda y: (y.values*x.values).sum()/(np.sqrt((y**2).sum())*np.sqrt((x**2).sum())),axis=1)/2


np.float64(0.74)

#### Problem 1
This is an open ended question that requires you to code. I have provided my own ratings for some of the movies in the dataset. What would you recommend to me based on my recommendations if you applied user-user filtering? Feel free to also change to your rankings. I ranked the top 5 movies according to the count of users who have ranked movies.

**Your solution here.**

In [11]:
data[['movieId','title']].value_counts()

,,count
movieId,title,
356,Forrest Gump (1994),329
318,"Shawshank Redemption, The (1994)",317
296,Pulp Fiction (1994),307
593,"Silence of the Lambs, The (1991)",279
2571,"Matrix, The (1999)",278
...,...,...
190207,Tilt (2011),1
190183,The Darkest Minds (2018),1
189713,BlacKkKlansman (2018),1


In [12]:
counts = data[['movieId','title']].value_counts().reset_index()

In [14]:
user_ratings = pd.DataFrame(index=['Dr. Anderson'],columns=counts['title'])
user_ratings.loc["Dr. Anderson","Forrest Gump (1994)"] = 4
user_ratings.loc["Dr. Anderson","Shawshank Redemption, The (1994)"] = 5
user_ratings.loc["Dr. Anderson","Pulp Fiction (1994)"] = 3
user_ratings.loc["Dr. Anderson","Silence of the Lambs, The (1991)"] = 2
user_ratings.loc["Dr. Anderson","Matrix, The (1999)"] = 5
user_ratings

title,Forrest Gump (1994),"Shawshank Redemption, The (1994)",Pulp Fiction (1994),"Silence of the Lambs, The (1991)","Matrix, The (1999)",Star Wars: Episode IV - A New Hope (1977),Jurassic Park (1993),Braveheart (1995),Terminator 2: Judgment Day (1991),Schindler's List (1993),...,"Game Over, Man! (2018)",Bunny (1998),Liquid Truth (2017),John From (2015),Jeff Ross Roasts the Border (2017),Tilt (2011),The Darkest Minds (2018),BlacKkKlansman (2018),Iron Soldier (2010),SuperFly (2018)
Dr. Anderson,4,5,3,2,5,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
ratings_reordered = ratings.unstack().T.loc[counts['movieId']].T # reorder the ratings to be the same as above
ratings_reordered.columns = user_ratings.columns
ratings_reordered

title,Forrest Gump (1994),"Shawshank Redemption, The (1994)",Pulp Fiction (1994),"Silence of the Lambs, The (1991)","Matrix, The (1999)",Star Wars: Episode IV - A New Hope (1977),Jurassic Park (1993),Braveheart (1995),Terminator 2: Judgment Day (1991),Schindler's List (1993),...,"Game Over, Man! (2018)",Bunny (1998),Liquid Truth (2017),John From (2015),Jeff Ross Roasts the Border (2017),Tilt (2011),The Darkest Minds (2018),BlacKkKlansman (2018),Iron Soldier (2010),SuperFly (2018)
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,3.0,4.0,5.0,5.0,4.0,4.0,NaN,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,1.0,5.0,1.0,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,3.0,5.0,NaN,NaN,NaN,NaN,4.0,3.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,4.0,3.5,5.0,4.5,5.0,4.5,2.5,3.5,3.5,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607,NaN,5.0,3.0,5.0,5.0,3.0,4.0,5.0,4.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,3.0,4.5,5.0,4.0,5.0,3.5,3.0,4.0,3.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
def recommend_user_user(data_raw,x_raw,N=10,frac=0.02):
  db = (data_raw.T-data_raw.T.mean()).fillna(0).T
  x = (x_raw - x_raw.mean()).fillna(0)

  candidate_movies = x_raw[x_raw.isna()].index
  preds = []
  for movie in x_raw:
      # Your solution here
      sims = db.loc[np.isnan(data_raw[movie])==False].apply(lambda y: (y.values*x.values).sum()/(np.sqrt((y**2).sum())*np.sqrt((x**2).sum())),axis=1)
      sims = sims.dropna()
      try:
          sorted_sims = sims.sort_values()[::-1]
      except:
          preds.append(0) # means there is no one that also rated this movie amongst all other users
          continue
      top_sims = sorted_sims.iloc[:N]
      ids = top_sims.index
      # Your solution here
      preds.append(db.loc[ids, movie].mean())
  return pd.Series(preds, index=candidate_movies)

mae = recommend_user_user(ratings_reordered,user_ratings)
mae

/tmp/ipykernel_203/1114736516.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x = (x_raw - x_raw.mean()).fillna(0)


#### Problem 2
Repeat problem 1 but recommend movies using item-item. Any difference? Which one do you think is more reasonable?

**Your solution here.**

In [ ]:

def predict_item_item(data_raw,x_raw,N=10,frac=0.02,debug={}):
    db = data_raw.drop(x_raw.name)
    db = (db.T-db.T.mean()).fillna(0).T
    db = db.T.fillna(0)
    candidate_movies = x_raw[x_raw.isna()].index
    preds = []
    for movie in x_raw:
        x = db.loc[movie]
        sims = db.drop(movie).loc[x_raw].apply(lambda y: (y.values*x.values).sum()/(np.sqrt((y**2).sum())*np.sqrt((x**2).sum())),axis=1)
        sims = sims.dropna()
        sorted_sims = sims.sort_values()[::-1]
        top_sims = sorted_sims.iloc[:N]
        ids = top_sims.index
        print(ids)
        preds.append(x_raw.loc[ids].mean())
    return pd.Series(preds, index=candidate_movies)
